# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. All datasets, record sets, fields, and columns are referenced by their Croissant `@id` to ensure clarity and reproducibility.

### Dataset Source
The dataset Croissant metadata can be found at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL for this dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
List the available record sets in the dataset and their Croissant `@id`s. Then, explore the fields (columns) available within the main record set.

In [ ]:
from pprint import pprint

# List all record sets in the dataset
print("Available record sets and their @id fields:")
for rs in metadata.record_sets:
    print(f"- name: {rs.name}\n  @id: {rs.id}\n  description: {rs.description}\n")

# For this dataset, let's inspect the first (and main) record set, and list its fields
main_recordset = metadata.record_sets[0]
print(f"Fields (columns) for record set '@id': {main_recordset.id}")
for field in main_recordset.fields:
    print(f"- {field.name} (@id: {field.id}, type: {field.data_type})")

## 3. Data Extraction
Load the main record set's entries as a DataFrame for further analysis. All references below use Croissant `@id` values.

In [ ]:
# Use the main record set's @id
main_recordset_id = main_recordset.id
record_sets_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")

# Display columns for the main record set
if main_recordset_id in dataframes:
    print(f"\nColumns in DataFrame ({main_recordset_id}):")
    print(dataframes[main_recordset_id].columns.tolist())
    display(dataframes[main_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)
As an example, let's process the 'MW [kDa]' (molecular weight) numeric field. We will filter for proteins where MW [kDa] > 20, normalize this field, and group by the 'Sample' field (if present).
All field references use their Croissant `@id`.

In [ ]:
# Identify the numeric field's Croissant @id, e.g., for MW [kDa]
# Find the @id for MW [kDa] (you can replace this logic as needed)
mw_field_id = None
sample_field_id = None

for field in main_recordset.fields:
    if 'MW' in field.name:
        mw_field_id = field.id
    if 'Sample' in field.name:
        sample_field_id = field.id

if mw_field_id is None:
    raise ValueError("Could not find MW field by name. Please check fields.")

# EDA steps
df = dataframes[main_recordset_id]

# Convert MW field to numeric if necessary
df[mw_field_id] = pd.to_numeric(df[mw_field_id], errors='coerce')

threshold = 20
filtered_df = df[df[mw_field_id] > threshold].copy()
print(f"Filtered records with {mw_field_id} > {threshold} (MW > 20 kDa): {len(filtered_df)} proteins")
display(filtered_df[[mw_field_id]].head())

# Normalize MW field
filtered_df[f"{mw_field_id}_normalized"] = (
    filtered_df[mw_field_id] - filtered_df[mw_field_id].mean()
) / filtered_df[mw_field_id].std()
print(f"\nNormalized MW for filtered proteins:")
display(filtered_df[[mw_field_id, f"{mw_field_id}_normalized"]].head())

# Group by Sample if present
if sample_field_id is not None and sample_field_id in filtered_df.columns:
    grouped = (
        filtered_df.groupby(sample_field_id)[mw_field_id]
        .mean()
        .reset_index()
        .rename(columns={mw_field_id: "Mean_MW_[kDa]"})
    )
    print(f"\nGrouped by Sample '{sample_field_id}':")
    display(grouped.head())

## 5. Visualization
Let's visualize the molecular weight distribution after filtering, and (if groupings are possible) the mean MW per sample.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[mw_field_id], bins=30, kde=True)
plt.title(f"Distribution of MW [kDa] for proteins with MW > {threshold}")
plt.xlabel('MW [kDa]')
plt.ylabel('Count')
plt.show()

# If groupings are possible, plot bar chart of mean MW by sample
if sample_field_id is not None and sample_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    grouped = (
        filtered_df.groupby(sample_field_id)[mw_field_id]
        .mean()
        .reset_index()
        .sort_values('Mean_MW_[kDa]', ascending=False)
    )
    sns.barplot(x=sample_field_id, y="Mean_MW_[kDa]", data=grouped)
    plt.title("Mean MW [kDa] per Sample")
    plt.ylabel("Mean MW [kDa]")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 6. Conclusion
This notebook showcased loading and exploring the FAIR² mass spectrometry protein dataset using the `mlcroissant` library. We:
- Loaded metadata and explored record set structure using Croissant `@id` references
- Loaded the main record set into a DataFrame for analysis
- Performed basic EDA: filtering by molecular weight, normalization, and grouping by sample
- Visualized distributions and group summaries

Further steps may include integrating with other omics datasets, advanced protein biomarker analysis, and workflow automation using Croissant's provenance features.